In [3]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import matplotlib.ticker as ticker
import matplotlib as mpl
import numpy as np
import matplotlib.patches as mpatches

# --------------------------------------------
# Step 1: Load timing data from .jsonl files
# --------------------------------------------

def load_jsonl_timings(label, path, file):
    file_path = os.path.join(path, file)
    rows = []
    if not os.path.exists(file_path):
        print(f"No file at: {file_path}")
        return rows

    with open(file_path) as f:
        for line in f:
            try:
                data = json.loads(line)
                rows.append({
                    "type": label,
                    "start_ms": int(data["start_ms"]),
                    "duration_ms": int(data["duration_ms"])
                })
            except Exception as e:
                print(f"Failed to parse line: {e}")
    return rows

# Matplotlib vector-friendly and font embedding settings
mpl.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "text.usetex": True,
    "pgf.texsystem": "pdflatex",
    "font.family": "sans-serif",
    "font.sans-serif": "Arial",
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "figure.titlesize": 10,
    "pgf.rcfonts": False
})

# Color Accessibility Pallette
custom_colors = ['#ffb000', '#f86c36', '#dc267f', '#9d62d6', '#648fff', '#00bfc4', '#90d200']
#custom_colors = ['#FFEB2E', '#FFB521', '#FF7028', '#E03A2F', '#DC267F', '#9D62D6', '#648FFF', '#00BFC4', '#4FE192', '#9EF572']

def autosize(n_labels):
    width = max(5.5, 3 + 0.75 * n_labels)
    height = 4.5
    return (width, height)

def save_plot(fig, path_without_ext):
    fig.savefig(f"{path_without_ext}.pdf", bbox_inches="tight")
    fig.savefig(f"{path_without_ext}.pgf", bbox_inches="tight")

def get_data_files(
                 config, 
                 output_prefix, 
                 data_file, 
                 custom_title_prefix=""
                ):

    rows = []
    for label, path in config.items():
        rows += load_jsonl_timings(label, path, data_file)

    df = pd.DataFrame(rows)
    if df.empty:
        print("⚠️ No timing data available. Skipping analysis.")
        return

    base_dir = f"experiments/{output_prefix}"
    os.makedirs(f"{base_dir}_plots", exist_ok=True)

    df.to_csv(f"{base_dir}_timings_summary.csv", index=False)
    print(f"✅ Saved timing summary to {base_dir}_timings_summary.csv")

    df["start_time"] = pd.to_datetime(df["start_ms"], unit="ms")
    summary_table = df.groupby("type")["duration_ms"].describe()
    print(f"\n📊 Summary Table ({output_prefix}):")
    print(summary_table)
    summary_table.to_csv(f"{base_dir}_timings_stats.csv")


def run_analysis(config, 
                 output_prefix, 
                 data_file, 
                 custom_title_prefix="", 
                 custom_colors= ['#ffb000', '#f86c36', '#dc267f', '#9d62d6', '#648fff', '#00bfc4', '#90d200']
                ):

    rows = []
    for label, path in config.items():
        rows += load_jsonl_timings(label, path, data_file)

    df = pd.DataFrame(rows)
    if df.empty:
        print("⚠️ No timing data available. Skipping analysis.")
        return

    base_dir = f"experiments/{output_prefix}"
    os.makedirs(f"{base_dir}_plots", exist_ok=True)

    df.to_csv(f"{base_dir}_timings_summary.csv", index=False)
    print(f"✅ Saved timing summary to {base_dir}_timings_summary.csv")

    df["start_time"] = pd.to_datetime(df["start_ms"], unit="ms")
    summary_table = df.groupby("type")["duration_ms"].describe()
    print(f"\n📊 Summary Table ({output_prefix}):")
    print(summary_table)
    summary_table.to_csv(f"{base_dir}_timings_stats.csv")

    n_labels = df["type"].nunique()
    size = autosize(n_labels)

    # # Histogram
    fig = plt.figure(figsize=size)
    sns.histplot(
        data=df,
        x="duration_ms",
        hue="type",
        element="step",
        bins=30,
        common_norm=False,
        stat="count",
        palette=custom_colors[:n_labels],
        legend=False
    )

    handles = [
        mpatches.Patch(color=custom_colors[i], label=label)
        for i, label in enumerate(df["type"].unique())
    ]
    plt.title(rf"\textbf{{{custom_title_prefix} Distribution of Execution Times}}")
    plt.xlabel("Duration (ms)")
    plt.ylabel("Count")
    plt.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=len(handles))
    plt.tight_layout()
    save_plot(fig, f"{base_dir}_plots/histogram")
    plt.show()
    plt.close()

    # Boxplot
    fig = plt.figure(figsize=size)
    sns.boxplot(data=df, x="type", y="duration_ms", palette=custom_colors)
    plt.title(rf"\textbf{{{custom_title_prefix} Duration Variation}}")
    plt.ylabel("Duration (ms)")
    plt.xlabel("")
    plt.xticks(rotation=30)
    plt.tight_layout()
    save_plot(fig, f"{base_dir}_plots/boxplot")
    plt.show()
    plt.close()

    # KDF Plot
    min_val = df["duration_ms"].min()
    max_val = df["duration_ms"].max()
    if min_val == max_val:
        print("All values are identical; KDE plot will not work.")
    else:
        plt.figure(figsize=size)
        sns.kdeplot(
            data=df,
            x="duration_ms",
            hue="type",
            fill=True,
            common_norm=False,
            linewidth=1.5,
            bw_adjust=0.7,
            palette=custom_colors,
            legend=False
        )

        handles = [
            mpatches.Patch(color=custom_colors[i], label=label)
            for i, label in enumerate(df["type"].unique())
        ]
    
        plt.xlabel("Duration (ms)")
        plt.ylabel("Density")
        plt.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=len(handles))
        plt.title(rf"\textbf{{{custom_title_prefix} KDE Plot of Execution Times}}")
        plt.tight_layout()
    
        fig = plt.gcf()  # get the current figure with the plot
        save_plot(fig, f"{base_dir}_plots/kde")
        plt.show()
        plt.close()


    # CDF
    fig = plt.figure(figsize=size) 
    for i, label in enumerate(df['type'].unique()):
        subset = df[df['type'] == label]['duration_ms'].sort_values()
        cum_prob = range(1, len(subset) + 1)
        plt.plot(subset, [x / len(subset) for x in cum_prob], label=label, color=custom_colors[i % len(custom_colors)])
    plt.title(rf"\textbf{{{custom_title_prefix} Cumulative Distribution}}")
    plt.xlabel("Duration (ms)")
    plt.ylabel("Cumulative Probability")
    plt.legend(title=None, loc='upper center',  bbox_to_anchor=(0.5, -0.15), ncol=n_labels)
    plt.tight_layout()
    save_plot(fig, f"{base_dir}_plots/cdf")
    plt.show()
    plt.close()

    # Trend
    df["elapsed_sec"] = df.groupby("type")["start_time"].transform(lambda x: (x - x.min()).dt.total_seconds())

    fig = plt.figure(figsize=size)
    for i, label in enumerate(df["type"].unique()):
        sub_df = df[df["type"] == label]
        plt.plot(
            sub_df["elapsed_sec"],
            sub_df["duration_ms"],
            label=label,
            linewidth=1.7,
            color=custom_colors[i % len(custom_colors)]
        )
    plt.legend(title=None, loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=n_labels)
    plt.title(rf"\textbf{{{custom_title_prefix} Execution Time Trends}}")
    plt.xlabel("Elapsed Time (s)")
    plt.ylabel("Duration (ms)")
    plt.tight_layout()
    save_plot(fig, f"{base_dir}_plots/trend")
    plt.show()
    plt.close()

    # Bar chart with error bars
    type_order = df["type"].drop_duplicates().tolist()
    means = df.groupby("type")["duration_ms"].mean().reindex(type_order)
    stds = df.groupby("type")["duration_ms"].std().reindex(type_order)
    fig = plt.figure(figsize=size)
    plt.bar(type_order, means.values, yerr=stds.values, capsize=4, color=custom_colors[:len(type_order)])
    plt.title(rf"\textbf{{{custom_title_prefix} Mean Duration with Std. Dev.}}")
    plt.ylabel("Duration (ms)")
    plt.xlabel("")
    plt.xticks(rotation=30)
    plt.tight_layout()
    save_plot(fig, f"{base_dir}_plots/means_with_error")
    plt.show()
    plt.close()

    # Outliers
    print(f"\n📉 Outliers in {output_prefix} (> 2 std dev):")
    for label in df['type'].unique():
        subset = df[df['type'] == label]
        mu = subset["duration_ms"].mean()
        sigma = subset["duration_ms"].std()
        outliers = subset[subset["duration_ms"] > mu + 2 * sigma]
        if not outliers.empty:
            print(f"\nOutliers in {label}:")
            print(outliers[["start_ms", "duration_ms"]])

def plot_core_subplots(core_paths, data_file="timings.jsonl", output_prefix="core_subplots"):
    # Load data
    rows = []
    for label, path in core_paths.items():
        rows += load_jsonl_timings(label, path, data_file)
    df = pd.DataFrame(rows)

    if df.empty:
        print("⚠️ No data for core subplots.")
        return

    # Setup figure
    fig, axes = plt.subplots(1, 3, figsize=(9, 3))  # 3 side-by-side plots
    plot_types = ["proof gen", "proof verify", "latency"]
    colors = custom_colors[:3]

    for i, label in enumerate(plot_types):
        ax = axes[i]
        subset = df[df["type"] == label]
        sns.histplot(
            data=subset,
            x="duration_ms",
            element="step",
            bins=30,
            common_norm=False,
            stat="count",
            ax=ax,
            color=colors[i]
        )
        ax.set_title(rf"\textbf{{{label}}}")
        ax.set_xlabel("Duration (ms)")
        if i == 0:
            ax.set_ylabel("Count")
        else:
            ax.set_ylabel("")

    plt.tight_layout()
    out_path = f"experiments/{output_prefix}"
    os.makedirs("experiments", exist_ok=True)
    fig.savefig(f"{out_path}.pdf", bbox_inches="tight")
    fig.savefig(f"{out_path}.pgf", bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"✅ Saved subplot plots to {out_path}.pdf and .pgf")

def plot_core_line_subplots(core_paths, data_file="timings.jsonl", output_prefix="core"):
    base_dir = f"experiments/{output_prefix}"
    # Load data
    rows = []
    for label, path in core_paths.items():
        rows += load_jsonl_timings(label, path, data_file)
    df = pd.DataFrame(rows)

    if df.empty:
        print("⚠️ No data for core line subplots.")
        return

    # Time conversion
    df["start_time"] = pd.to_datetime(df["start_ms"], unit="ms")
    df["elapsed_sec"] = df.groupby("type")["start_time"].transform(lambda x: (x - x.min()).dt.total_seconds())

    # Setup vertical subplots
    fig, axes = plt.subplots(3, 1, figsize=(6, 8), sharex=True)
    fig.subplots_adjust(hspace=0.3)
    plot_types = ["proof gen", "latency", "proof verify"]
    colors = custom_colors[:3]

    for i, label in enumerate(plot_types):
        ax = axes[i]
        sub_df = df[df["type"] == label]
        ax.plot(
            sub_df["elapsed_sec"],
            sub_df["duration_ms"],
            label=label,
            linewidth=1.2,
            color=colors[i]
        )
        cap_label = label.title()
        ax.set_title(rf"\textbf{{Anonymous Message {cap_label}}}")
        ax.grid(True, linestyle="--", alpha=0.5)

    # Common y-label
    fig.text(0.04, 0.5, 'Duration (ms)', va='center', rotation='vertical', fontsize=12)
    fig.supxlabel("Elapsed Time (s)", y=0.08, va='center', fontsize=12)

    fig.suptitle(rf"\textbf{{Anonymous Message Proof Generation, Proof Verification, and Latency }}", x=0.56)

    # Collect all handles/labels across subplots
    handles, labels = [], []
    for ax in axes:
        h, l = ax.get_legend_handles_labels()
        handles.extend(h)
        labels.extend(l)
    
    # Optional: deduplicate
    seen = set()
    unique = [(h, l) for h, l in zip(handles, labels) if not (l in seen or seen.add(l))]
    handles, labels = zip(*unique)
    
    # Shared legend at bottom
    fig.legend(handles, labels, loc='lower center', ncol=len(labels), bbox_to_anchor=(0.5, -0.003))

    # Save plots
    plt.tight_layout(rect=[0.05, 0.05, 1, 0.98])
    out_path = f"experiments/{output_prefix}"
    os.makedirs("experiments", exist_ok=True)
    save_plot(fig, f"{base_dir}_plots/subplot")
    plt.show()
    plt.close()
    print(f"✅ Saved stacked line subplots to {out_path}.pdf and .pgf")

file_configs = [
    {
        "name": "features",
        "paths": {
            "Anon": "json_files/anon_msg",
            "Lim. Pseudonym": "json_files/rate_pseudo",
            "Unlim. Pseudonym": "json_files/pseudo_msg",
            "Authorship": "json_files/author",
            "Vote": "json_files/pseudo_vote",
            "Badge": "json_files/badge", 
            "Scan": "json_files/scan", 
        },
        "colors": ['#69C1F5', '#F7EB5E', '#FFBA46', '#EE6677', '#A365C7'],
        "data_file": "timings.jsonl",
        "title_prefix": "Feature Proof Generation"
    },
    {
        "name": "latency_features",
        "paths": {
            "Anon": "json_files/anon_msg",
            "Lim. Pseudonym": "json_files/rate_pseudo",
            "Unlim. Pseudonym": "json_files/pseudo_msg",
            "Authorship": "json_files/author",
            "Vote": "json_files/pseudo_vote",
            "Badge": "json_files/badge",
            "Scan": "json_files/scan", 
        },
        "colors": ['#69C1F5', '#F7EB5E', '#FFBA46', '#EE6677', '#A365C7'],
        "data_file": "features_timings.jsonl",
        "title_prefix": "Feature Latency"
    },
    {
        "name": "verify_features",
        "paths": {
            "Anon": "json_files/anon_msg",
            "Lim. Pseudonym": "json_files/rate_pseudo",
            "Unlim. Pseudonym": "json_files/pseudo_msg",
            "Authorship": "json_files/author",
            "Vote": "json_files/pseudo_vote",
            "Badge": "json_files/badge",
            "Scan": "json_files/scan", 
        },
        "colors": ['#69C1F5', '#F7EB5E', '#FFBA46', '#EE6677', '#A365C7'],
        "data_file": "verify_timings.jsonl",
        "title_prefix": "Feature Proof Verification"
    },
    {
        "name": "rep",
        "paths": {
            "rep": "json_files/rep"
        },
        "colors": ['#009E73'],
        "data_file": "features_timings.jsonl",
        "title_prefix": "Reputation Feature"
    },
    {
        "name": "call_cb",
        "paths": {
            "rep": "json_files/rep"
        },
        "colors": ['#E69F00'],
        "data_file": "call_timings.jsonl",
        "title_prefix": "Post Callback Timing"
    },
    {
        "name": "epoch",
        "paths": {
            "rep": "json_files/rep"
        },
        "colors": ['#CC79A7'],
        "data_file": "epoch_timings.jsonl",
        "title_prefix": "Update Epoch Timing"
    }
]

for config in file_configs:
    get_data_files(
        config["paths"],
        output_prefix=config["name"],
        data_file=config["data_file"],
        custom_title_prefix=config["title_prefix"]
    )


✅ Saved timing summary to experiments/features_timings_summary.csv

📊 Summary Table (features):
                  count        mean        std    min     25%    50%     75%  \
type                                                                           
Anon               70.0  128.700000   6.616514  116.0  124.25  128.0  131.00   
Authorship          5.0  112.200000   5.019960  105.0  109.00  115.0  115.00   
Badge              13.0  102.846154   4.412976   95.0  101.00  105.0  106.00   
Lim. Pseudonym     10.0  129.500000  11.157957  113.0  123.50  126.5  131.75   
Scan               48.0  214.187500  13.907990  199.0  210.00  213.0  215.00   
Unlim. Pseudonym   40.0  131.650000   7.566966  116.0  128.00  130.5  134.00   
Vote               10.0  109.200000   4.049691  103.0  107.25  108.5  111.50   

                    max  
type                     
Anon              147.0  
Authorship        117.0  
Badge             107.0  
Lim. Pseudonym    151.0  
Scan              302.0  
U